<a href="https://colab.research.google.com/github/fsananna/3_Cloud/blob/main/IEEEpaperSemevalCodeAnanna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
# --- BLOCK A: DATA PIPELINE ---
from google.colab import drive
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from pathlib import Path
import ast

# 1. Mount Drive
drive.mount('/content/drive')

# 2. CONFIGURATION (Your Real Paths)
BASE_DATA_DIR = Path('/content/drive/MyDrive/Thesis2024_Farhana/AllData')
PROCESSED_DIR = BASE_DATA_DIR / 'Processed_Features'

# TSV Paths (Your Real Splits)
SPLITS = {
    'train': BASE_DATA_DIR / 'SubtaskA_Train' / 'train' / 'subtask_a_train.tsv',
    'dev':   BASE_DATA_DIR / 'SubtaskA_Dev'   / 'dev'   / 'subtask_a_dev.tsv',
    'test':  BASE_DATA_DIR / 'SubtaskA_Test'  / 'test'  / 'subtask_a_test.tsv'
}

class IdiomRankDataset(Dataset):
    def __init__(self, split_name, use_tagged_text=False):
        """
        Args:
            split_name (str): 'train', 'dev', or 'test'
            use_tagged_text (bool):
                If False, loads from 'text_embeddings_plain'
                If True,  loads from 'text_embeddings_tagged' (With Sentence Type)
        """
        self.split_name = split_name
        self.tsv_path = SPLITS[split_name]

        # Load the dataframe
        self.df = pd.read_csv(self.tsv_path, sep='\t', dtype=str).fillna("")

        # Output With and Without Sentence Type
        if use_tagged_text:
            self.text_dir = PROCESSED_DIR / 'text_embeddings_tagged'
            print(f"[{split_name.upper()}] Loading TAGGED text (With Sentence Type)")
        else:
            self.text_dir = PROCESSED_DIR / 'text_embeddings_plain'
            print(f"[{split_name.upper()}] Loading PLAIN text (Without Sentence Type)")

        self.image_dir = PROCESSED_DIR / 'image_features'

        # Relevance Mapping: Rank 1 -> Score 1.0, Rank 2 -> Score 0.8, etc.
        self.rank_to_score = {0: 1.0, 1: 0.8, 2: 0.6, 3: 0.4, 4: 0.2}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sample_id = f"{self.split_name}_{idx}"

        # 1. Load Text Embedding
        text_path = self.text_dir / f"{sample_id}.pt"
        if text_path.exists():
            text_emb = torch.load(text_path).squeeze() # Shape: [768]
        else:
            text_emb = torch.zeros(768) # Safety fallback

        # 2. Load 5 Image Embeddings
        idiom_folder = str(row['compound']).strip()
        image_feats = []
        image_names = []

        for i in range(1, 6):
            fname = str(row[f'image{i}_name']).strip()
            image_names.append(fname)

            img_path = self.image_dir / idiom_folder / f"{fname}.pt"
            if img_path.exists():
                feat = torch.load(img_path).squeeze() # Shape: [2048]
            else:
                feat = torch.zeros(2048)
            image_feats.append(feat)

        images_tensor = torch.stack(image_feats) # Shape: [5, 2048]

        # 3. Create Target Scores (Ground Truth)
        target_scores = self._create_target_scores(row['expected_order'], image_names)

        return text_emb, images_tensor, target_scores

    def _create_target_scores(self, expected_order_str, image_names):
        """Converts filename list ['a.png', 'b.png'] into scores [1.0, 0.8]"""
        try:
            gt_order = ast.literal_eval(str(expected_order_str))
        except:
            gt_order = []

        scores = torch.zeros(5)

        # Assign scores based on the rank in the Ground Truth list
        for rank, filename in enumerate(gt_order):
            if rank > 4: break
            if filename in image_names:
                # Find which position (0-4) this file is in our input list
                idx_in_input = image_names.index(filename)
                scores[idx_in_input] = self.rank_to_score[rank]

        return scores

# --- TEST BLOCK (Verify the Switch Works) ---
if __name__ == "__main__":
    print("Testing Option A (Plain)...")
    ds_plain = IdiomRankDataset('train', use_tagged_text=False)

    print("\nTesting Option B (Tagged)...")
    ds_tagged = IdiomRankDataset('train', use_tagged_text=True)

    # Check first sample
    t, i, s = ds_plain[0]
    print(f"\nSample 0 loaded successfully.")
    print(f"Text Shape: {t.shape} | Images Shape: {i.shape} | Targets: {s}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Testing Option A (Plain)...
[TRAIN] Loading PLAIN text (Without Sentence Type)

Testing Option B (Tagged)...
[TRAIN] Loading TAGGED text (With Sentence Type)

Sample 0 loaded successfully.
Text Shape: torch.Size([768]) | Images Shape: torch.Size([5, 2048]) | Targets: tensor([1.0000, 0.6000, 0.8000, 0.2000, 0.4000])


In [26]:
# --- BLOCK B: MODEL ARCHITECTURE (Siamese Attention + LSTM) ---
import torch
import torch.nn as nn

class SiameseAttentionRanker(nn.Module):
    def __init__(self, text_dim=768, img_dim=2048, hidden_dim=512):
        super(SiameseAttentionRanker, self).__init__()

        # 1. Projectors: Squash BERT and ResNet to the same size (512)
        self.text_projector = nn.Linear(text_dim, hidden_dim)
        self.img_projector = nn.Linear(img_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.1) # Helps prevent overfitting

        # 2. Siamese Attention Layer
        # We define ONE layer, but we use it 5 times (Shared Weights)
        # This is what makes it a "Siamese" network.
        self.attention = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=4, batch_first=True)

        # 3. Bi-Directional LSTM
        # Compares the 5 images against each other
        # Output dim will be hidden_dim (256 * 2 directions = 512)
        self.lstm = nn.LSTM(input_size=hidden_dim,
                            hidden_size=hidden_dim // 2,
                            batch_first=True,
                            bidirectional=True)

        # 4. Scoring Head
        # Takes the LSTM output and gives a score (0 to 1)
        self.score_head = nn.Linear(hidden_dim, 1)

    def forward(self, text_emb, image_embs):
        """
        Args:
            text_emb:   [Batch, 768]
            image_embs: [Batch, 5, 2048]
        Returns:
            scores:     [Batch, 5]
        """
        batch_size = text_emb.size(0)

        # --- Step 1: Projection ---
        # Squash inputs to 512 dim
        t = self.relu(self.text_projector(text_emb))        # [Batch, 512]
        i = self.relu(self.img_projector(image_embs))       # [Batch, 5, 512]

        # --- Step 2: The Siamese Loop (Individual Attention) ---

        # Reshape text to [Batch, 1, 512] to act as the "Query"
        t_query = t.unsqueeze(1)

        attended_features_list = []

        # Loop through each of the 5 images
        for k in range(5):
            # Extract the k-th image. Shape: [Batch, 1, 512]
            img_k = i[:, k:k+1, :]

            # Apply Attention: Text looks at THIS image only
            # Query=Text, Key=Image_k, Value=Image_k
            attn_out, _ = self.attention(query=t_query, key=img_k, value=img_k)

            # Residual Connection (Add original image back to result)
            # This helps the model remember the original visual features
            fused_k = img_k + attn_out

            attended_features_list.append(fused_k)

        # Stack them back into a sequence: [Batch, 5, 512]
        # Now we have a sequence of 5 "Context-Aware" Image features
        fused_sequence = torch.cat(attended_features_list, dim=1)
        fused_sequence = self.dropout(fused_sequence)

        # --- Step 3: Contextual Ranking (LSTM) ---
        # The LSTM looks at the whole group of 5 to make final comparisons
        lstm_out, _ = self.lstm(fused_sequence) # Shape: [Batch, 5, 512]

        # --- Step 4: Final Scoring ---
        # Project down to 1 number per image
        scores = self.score_head(lstm_out)      # Shape: [Batch, 5, 1]

        return scores.squeeze(-1)               # Shape: [Batch, 5]

# --- TEST BLOCK (Verify Architecture) ---
if __name__ == "__main__":
    print("Initializing Siamese Model...")
    model = SiameseAttentionRanker()

    # Create dummy data matching your Dataset output
    fake_text = torch.rand(16, 768)      # Batch of 16 texts
    fake_imgs = torch.rand(16, 5, 2048)  # Batch of 16 sets of images

    print("Running Forward Pass...")
    output = model(fake_text, fake_imgs)

    print("\n✅ MODEL ARCHITECTURE PASSED!")
    print(f"Input Shapes: Text {fake_text.shape}, Images {fake_imgs.shape}")
    print(f"Output Shape: {output.shape}") # Should be [16, 5]
    print("The model is ready for training.")

Initializing Siamese Model...
Running Forward Pass...

✅ MODEL ARCHITECTURE PASSED!
Input Shapes: Text torch.Size([16, 768]), Images torch.Size([16, 5, 2048])
Output Shape: torch.Size([16, 5])
The model is ready for training.


In [28]:
# --- BLOCK C & D: TRAIN-VAL SPLIT EXPERIMENT ---
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import numpy as np
import os
import matplotlib.pyplot as plt

# --- 1. METRIC FUNCTIONS ---
def dcg_score(y_true, y_score, k=5):
    order = np.argsort(y_score)[::-1]
    y_true = np.take(y_true, order[:k])
    gains = 2 ** y_true - 1
    discounts = np.log2(np.arange(len(y_true)) + 2)
    return np.sum(gains / discounts)

def ndcg_score(y_true, y_score, k=5):
    best = dcg_score(y_true, y_true, k)
    actual = dcg_score(y_true, y_score, k)
    return actual / best if best > 0 else 0.0

# --- 2. THE EXPERIMENT ENGINE ---
def run_scientific_experiment(use_tagged_text=False, epochs=15, batch_size=16):

    # A. Load the ONLY dataset with answers (TRAIN)
    full_dataset = IdiomRankDataset('train', use_tagged_text=use_tagged_text)

    # B. Create a Scientific Split (80% Train, 20% Validation)
    # We set a 'seed' so the split is exactly the same for Experiment A and B
    generator = torch.Generator().manual_seed(42)

    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_ds, val_ds = random_split(full_dataset, [train_size, val_size], generator=generator)

    print(f"\n SETUP: Tags={use_tagged_text}")
    print(f"   Total Samples: {len(full_dataset)}")
    print(f"   Training on:   {len(train_ds)} samples")
    print(f"   Validating on: {len(val_ds)} samples (Hidden Hold-out)")

    # C. Dataloaders
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False) # Batch=1 for accurate metrics

    # D. Model Setup
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = SiameseAttentionRanker().to(device)
    criterion = nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=0.0001)

    # E. Training Loop
    print("Training Started...")
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for t, i, targets in train_loader:
            optimizer.zero_grad()
            preds = model(t.to(device), i.to(device))
            loss = criterion(preds, targets.to(device))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

    # F. Evaluation Loop (On the 20% Hold-out Set)
    print(" Evaluating on Hold-out Set...")
    model.eval()
    total_ndcg = 0
    total_top1 = 0

    with torch.no_grad():
        for t, i, targets in val_loader:
            preds = model(t.to(device), i.to(device)).cpu().numpy().flatten()
            truth = targets.numpy().flatten()

            # Metrics
            total_ndcg += ndcg_score(truth, preds)
            if np.argmax(preds) == np.argmax(truth):
                total_top1 += 1

    # G. Final Results
    avg_ndcg = total_ndcg / len(val_loader)
    avg_top1 = total_top1 / len(val_loader)

    print(f"  RESULT: Accuracy={avg_top1:.4f} | NDCG={avg_ndcg:.4f}")
    return avg_top1, avg_ndcg

# --- 3. RUN THE COMPARISON ---

print("=======================================")
print("Output Without Sentence Type")
print("=======================================")
acc_a, ndcg_a = run_scientific_experiment(use_tagged_text=False)

print("\n\n=======================================")
print("Output With Sentence Type")
print("=======================================")
acc_b, ndcg_b = run_scientific_experiment(use_tagged_text=True)

# --- 4. THE PAPER TABLE ---
print("\n\n################################################")
print("           FINAL RESULTS                ")
print("   (Evaluated on 20% Hold-out Training Data)    ")
print("################################################")
print(f"Output Without Sentence Type | Acc: {acc_a:.4f} | NDCG: {ndcg_a:.4f}")
print(f"Output With Sentence Type | Acc: {acc_b:.4f} | NDCG: {ndcg_b:.4f}")
print("################################################")

Output Without Sentence Type
[TRAIN] Loading PLAIN text (Without Sentence Type)

 SETUP: Tags=False
   Total Samples: 70
   Training on:   56 samples
   Validating on: 14 samples (Hidden Hold-out)
Training Started...
 Evaluating on Hold-out Set...
  RESULT: Accuracy=0.2857 | NDCG=0.8673


Output With Sentence Type
[TRAIN] Loading TAGGED text (With Sentence Type)

 SETUP: Tags=True
   Total Samples: 70
   Training on:   56 samples
   Validating on: 14 samples (Hidden Hold-out)
Training Started...
 Evaluating on Hold-out Set...
  RESULT: Accuracy=0.2857 | NDCG=0.8641


################################################
           FINAL RESULTS                
   (Evaluated on 20% Hold-out Training Data)    
################################################
Output Without Sentence Type | Acc: 0.2857 | NDCG: 0.8673
Output With Sentence Type | Acc: 0.2857 | NDCG: 0.8641
################################################
